# Draw Event Rate Plots with Syst Unc

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
import joblib

from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *
from analysis_village.unfolding.covariance import *
from analysis_village.unfolding.variable_configs import VariableConfig

from analysis_village.cohpi.unfolding.cohpi_topologies import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

## Open joblib file : output of ana_syst_dfs.ipynb

In [ ]:
data_loaded = joblib.load('multisim_objects.joblib')

p_mu_objs = data_loaded["p_mu_objs"]
cos_mu_objs = data_loaded["cos_mu_objs"]
cos_pi_objs = data_loaded["cos_pi_objs"]
data_tot_pot = data_loaded["data_tot_pot"]

For data
- sr_bnb_light_events, sb_bnb_light_events
- sr_offbeam_light_events, sb_offbeam_light_events


For MC bnb + cosmic
- {reg}\_signal\_univ\_events\_{sys_label}\_{cov_type}
- {reg}\_signal\_sel\_reco_cv\_{cov_type}
- {reg}\_bkg\_univ\_events\_{sys_label}\_{cov_type}
- {reg}\_bkg\_sel\_reco\_cv\_{cov_type}


## Define multiverse knobs again for loops

### GENIE

In [ ]:
## for NormCCCOH, using 0p5

genie_multisigma_knobs = [
                            # - QE
                            "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",

                            # - MEC
                            "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",

                            # - RES
                            "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
                            "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
                            "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
                            "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",

                            # - DIS
                            "GENIEReWeight_SBN_v1_multisigma_AhtBY",
                            "GENIEReWeight_SBN_v1_multisigma_BhtBY",
                            "GENIEReWeight_SBN_v1_multisigma_CV1uBY",
                            "GENIEReWeight_SBN_v1_multisigma_CV2uBY",

                            # - COH
                            "GENIEReWeight_SBN_v1_multisigma_NormCCCOH_0p5",
                            "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",

                            # - NC Elas
                            "GENIEReWeight_SBN_v1_multisigma_MaNCEL",
                            "GENIEReWeight_SBN_v1_multisigma_EtaNCEL",

                            # - FSI
                            "GENIEReWeight_SBN_v1_multisigma_MFP_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrCEx_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrInel_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrAbs_pi",
                            "GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi",
                            ## -- In AR23 only
                            "GENIEReWeight_SBN_v1_multisigma_MFP_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrCEx_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrInel_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrAbs_N",
                            "GENIEReWeight_SBN_v1_multisigma_FrPiProd_N",
                          ]

genie_multisim_knobs = [
                            # - QE
                            "GENIEReWeight_SBN_v1_multisim_CoulombCCQE",

                            # - MEC
                            "GENIEReWeight_SBN_v1_multisim_NormCCMEC",
                            "GENIEReWeight_SBN_v1_multisim_NormNCMEC",

                            # - RES
                            "GENIEReWeight_SBN_v1_multisim_RDecBR1eta",
                            "GENIEReWeight_SBN_v1_multisim_RDecBR1gamma",

                            # - Non-res pion
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi",
                            "GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi",
                        ]

In [ ]:
genie_all_knobs = genie_multisim_knobs + genie_multisigma_knobs

In [ ]:
genie_all_knobs

### Flux

In [ ]:
flux_knobs = ['expskin_Flux', 'horncurrent_Flux', 'kminus_Flux', 'kplus_Flux', 'kzero_Flux', 'nucleoninexsec_Flux', 'nucleonqexsec_Flux', 'nucleontotxsec_Flux', 'piminus_Flux', 'pioninexsec_Flux', 'pionqexsec_Flux', 'piontotxsec_Flux', 'piplus_Flux'] 

### G4

In [ ]:
g4_knobs = ['reinteractions_kminus_Geant4', 'reinteractions_kplus_Geant4', 'reinteractions_neutron_Geant4', 'reinteractions_piminus_Geant4', 'reinteractions_piplus_Geant4', 'reinteractions_proton_Geant4']

## Collect Cov matrices for all knobs

In [ ]:
p_mu_objs.keys()

In [ ]:
cos_mu_objs['sb_offbeam_light_events']

In [ ]:
def make_rate_total_cov_for_knob(knob, obj):
    obj['sr_total_bkg_cv_rate'] = obj['sr_bkg_sel_reco_cv_rate'].sum(axis=0)
    obj['sr_total_bkg_univ_events_' + knob + '_rate'] = obj['sr_bkg_univ_events_' + knob + '_rate'].sum(axis=1)
    obj['sr_total_mc_cv_rate'] = obj['sr_total_bkg_cv_rate'] + obj['sr_signal_sel_reco_cv_rate']
    obj['sr_total_mc_univ_events_' + knob + '_rate'] = obj['sr_total_bkg_univ_events_' + knob + '_rate'] + obj['sr_signal_univ_events_' + knob + '_rate']
    obj['sr_total_mc_evt_cov_' + knob] = get_covariance_matrix_self(obj['sr_total_mc_univ_events_' + knob + '_rate'], obj['sr_total_mc_cv_rate'])

    obj['sb_total_bkg_cv_rate'] = obj['sb_bkg_sel_reco_cv_rate'].sum(axis=0)
    obj['sb_total_bkg_univ_events_' + knob + '_rate'] = obj['sb_bkg_univ_events_' + knob + '_rate'].sum(axis=1)
    obj['sb_total_mc_cv_rate'] = obj['sb_total_bkg_cv_rate'] + obj['sb_signal_sel_reco_cv_rate']
    obj['sb_total_mc_univ_events_' + knob + '_rate'] = obj['sb_total_bkg_univ_events_' + knob + '_rate'] + obj['sb_signal_univ_events_' + knob + '_rate']
    obj['sb_total_mc_evt_cov_' + knob] = get_covariance_matrix_self(obj['sb_total_mc_univ_events_' + knob + '_rate'], obj['sb_total_mc_cv_rate'])

def make_covs_for_obj(obj):
    for knob in genie_all_knobs:
        make_rate_total_cov_for_knob(knob, obj)

    for knob in flux_knobs:
        make_rate_total_cov_for_knob(knob, obj)

    for knob in g4_knobs:
        make_rate_total_cov_for_knob(knob, obj)

    make_rate_total_cov_for_knob("GENIE_NormCCCOH_0p5", obj)
    make_rate_total_cov_for_knob("Flux", obj)
    make_rate_total_cov_for_knob("G4", obj)
    make_rate_total_cov_for_knob("MCstat", obj)

    # make cov for offbeam
    obj['sr_offbeam_light_events_cov'] = np.diag(obj['sr_offbeam_light_events'])
    obj['sb_offbeam_light_events_cov'] = np.diag(obj['sb_offbeam_light_events'])

    # add flat covs for total MC
    ## -- POT 2%
    obj['sr_total_mc_evt_cov_POT'] = {}
    obj['sr_total_mc_evt_cov_POT']['cov'] = np.diag((0.02 * obj['sr_total_mc_cv_rate'])**2)
    obj['sb_total_mc_evt_cov_POT'] = {}
    obj['sb_total_mc_evt_cov_POT']['cov'] = np.diag((0.02 * obj['sb_total_mc_cv_rate'])**2)
    ## -- Argon target 1%
    obj['sr_total_mc_evt_cov_Ar'] = {}
    obj['sr_total_mc_evt_cov_Ar']['cov'] = np.diag((0.01 * obj['sr_total_mc_cv_rate'])**2)
    obj['sb_total_mc_evt_cov_Ar'] = {}
    obj['sb_total_mc_evt_cov_Ar']['cov'] = np.diag((0.01 * obj['sb_total_mc_cv_rate'])**2)
    ## -- detector systematics flat 10%
    obj['sr_total_mc_evt_cov_Det'] = {}
    obj['sr_total_mc_evt_cov_Det']['cov'] = np.diag((0.1 * obj['sr_total_mc_cv_rate'])**2)
    obj['sb_total_mc_evt_cov_Det'] = {}
    obj['sb_total_mc_evt_cov_Det']['cov'] = np.diag((0.1 * obj['sb_total_mc_cv_rate'])**2)



In [ ]:
make_covs_for_obj(p_mu_objs)
make_covs_for_obj(cos_mu_objs)
make_covs_for_obj(cos_pi_objs)

In [ ]:
p_mu_objs['sr_total_mc_evt_cov_GENIE_NormCCCOH_0p5']

## Draw Data vs. MC plots with total syst and unc. budget

### plotters

In [ ]:
import matplotlib.gridspec as gridspec
from scipy.stats import chi2
def draw_data_vs_mc_rate(obj, reg='sr', is_logx=False, is_logy=False, sample_str="", region_str="Signal Region", xlim=None, ratio_ylim=(0.0, 2.0), zoom_xrange=None, zoom_pos=[0.55, 0.35, 0.4, 0.4]):
    fig = plt.figure(figsize=(8, 8), dpi=100)
    gs = gridspec.GridSpec(2, 1, height_ratios=[5, 1], hspace=0.10)
    ax_main = fig.add_subplot(gs[0])
    ax_ratio = fig.add_subplot(gs[1], sharex=ax_main)

    if is_logx:
        ax_main.set_xscale('log')
        ax_ratio.set_xscale('log')        
    if is_logy:
        ax_main.set_yscale('log')

    # Main plot styling
    ax_main.set_xlabel("")  
    ax_main.set_ylabel(f"Events/bin ({data_tot_pot:.2e} POT)", fontsize=20)
    ax_main.tick_params(width=2, length=10)
    for spine in ax_main.spines.values():
        spine.set_linewidth(2)
    plt.setp(ax_main.get_xticklabels(), visible=False)

    # SBND text on top
    title_text = "SBND Internal" if not sample_str else f"SBND {sample_str}, Internal"
    ax_main.text(0.00, 1.02, title_text, transform=ax_main.transAxes, fontsize=14, fontweight='bold')
    ax_main.text(1.0, 1.02, region_str, transform=ax_main.transAxes, fontsize=14, fontweight='bold', ha='right')

    # Ratio plot styling
    ax_ratio.axhline(1.0, color='red', linestyle='--', linewidth=1)
    ax_ratio.set_ylabel("Data/MC", fontsize=20)
    ax_ratio.set_xlabel(obj['varcfg'].var_labels[0], fontsize=20)
    ax_ratio.set_ylim(ratio_ylim)
    ax_ratio.tick_params(width=2, length=6)
    ax_ratio.minorticks_on()
    for spine in ax_ratio.spines.values():
        spine.set_linewidth(2)

    # ==========================================
    # Data Preparation
    # ==========================================
    bin_edges = obj['varcfg'].bins
    if xlim is None:
        ax_main.set_xlim(bin_edges[0], bin_edges[-1])
    else:
        ax_main.set_xlim(xlim)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Fetch Data
    this_signal = obj[reg + '_signal_sel_reco_cv_rate']
    this_bkg = obj[reg + '_bkg_sel_reco_cv_rate']
    this_data = obj[reg + '_bnb_light_events']
    this_offbeam = obj[reg + '_offbeam_light_events']

    # Combine all MC and Offbeam
    combined_mc = np.vstack((this_signal, this_bkg))
    full_mc_stack = np.vstack((combined_mc, this_offbeam))
    total_mc = np.sum(full_mc_stack, axis=0)
    
    # Calculate global totals for the legend
    total_mc_sum = np.sum(total_mc)
    total_data_sum = np.sum(this_data)

    # ==========================================
    # Covariance and Chi-Square Calculation
    # ==========================================
    cov_knobs = ["GENIE_NormCCCOH_0p5", "Flux", "G4", "MCstat", "POT", "Ar", "Det"]
    first_key = f"{reg}_total_mc_evt_cov_{cov_knobs[0]}"
    
    total_cov_sys_mc = np.zeros_like(obj[first_key]['cov'])
    for knob in cov_knobs:
        total_cov_sys_mc += obj[f"{reg}_total_mc_evt_cov_{knob}"]['cov']
    total_cov_sys_mc += obj[f"{reg}_offbeam_light_events_cov"]
    
    mc_err = np.sqrt(np.diag(total_cov_sys_mc))

    data_variance = np.where(this_data == 0, 1.0, this_data)
    total_cov_chi2 = total_cov_sys_mc + np.diag(data_variance)
    delta = this_data - total_mc
    inv_cov = np.linalg.pinv(total_cov_chi2) 
    this_chi2 = delta.T @ inv_cov @ delta
    ndf = len(this_data)
    p_value = chi2.sf(this_chi2, df=ndf)
    
    # Positioned slightly higher so it doesn't overlap the default zoom inset
    chi2_text = f"$\\chi^2/ndf = {this_chi2:.2f}/{ndf}$\n(p-value = {p_value:.3f})"
    ax_main.text(0.7, 0.7, chi2_text, transform=ax_main.transAxes, fontsize=12, ha='right', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    # ==========================================
    # Plotting
    # ==========================================

    # 1. Draw Stacked MC 
    mc_labels_with_fractions = [
        f"{label} ({(np.sum(mc_row)/total_mc_sum)*100:.1f}%)" if total_mc_sum > 0 else f"{label} (0.0%)"
        for label, mc_row in zip(mode_labels_with_offbeam, full_mc_stack)
    ]
    
    hist_data = [bin_centers] * len(full_mc_stack)
    ax_main.hist(hist_data, bins=bin_edges, weights=list(full_mc_stack), 
                 stacked=True, color=colors, label=mc_labels_with_fractions, 
                 histtype='stepfilled', edgecolor='none', linewidth=0)

    # 2. Add Error Band to Main Plot
    ax_main.bar(bin_centers, 2 * mc_err, bottom=total_mc - mc_err, 
                width=np.diff(bin_edges), facecolor='none', 
                edgecolor='black', hatch='xxxx', linewidth=0.0, 
                label=f'Total Unc. ({total_mc_sum:.1f})', zorder=5)

    # 3. Draw Data with Asymmetric Errors
    data_eylow, data_eyhigh = sh.return_data_stat_err(this_data)
    
    ax_main.errorbar(bin_centers, this_data, 
                     yerr=np.vstack((data_eylow, data_eyhigh)), 
                     fmt='o', color='black', label=f'Data ({total_data_sum:.0f})', 
                     markersize=5, capsize=3, linewidth=1.5, zorder=10)

    # Set appropriate y-limits dynamically
    max_y = max(np.max(total_mc), np.max(this_data + data_eyhigh))
    if is_logy:
        ax_main.set_ylim(0.01, max_y * 600)
    else:
        ax_main.set_ylim(0.0, max_y * 1.7)

    ax_main.legend(loc='upper left', fontsize=9, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

    # ==========================================
    # Draw Zoom Inset (If requested)
    # ==========================================
    if zoom_xrange is not None:
        ax_inset = ax_main.inset_axes(zoom_pos)
        
        if is_logx: ax_inset.set_xscale('log')
        if is_logy: ax_inset.set_yscale('log')

        # Redraw MC, Error Band, and Data on the inset
        ax_inset.hist(hist_data, bins=bin_edges, weights=list(full_mc_stack), 
                      stacked=True, color=colors, histtype='stepfilled', edgecolor='none', linewidth=0)
        
        ax_inset.bar(bin_centers, 2 * mc_err, bottom=total_mc - mc_err, 
                     width=np.diff(bin_edges), facecolor='none', edgecolor='black', hatch='xxxx', linewidth=0.0, zorder=5)
        
        ax_inset.errorbar(bin_centers, this_data, yerr=np.vstack((data_eylow, data_eyhigh)), 
                          fmt='o', color='black', markersize=4, capsize=2, linewidth=1.0, zorder=10)
        
        # Set inset x-limits
        ax_inset.set_xlim(zoom_xrange)
        
        # Determine dynamic y-limits strictly for the zoomed region
        zoom_mask = (bin_centers >= zoom_xrange[0]) & (bin_centers <= zoom_xrange[1])
        if np.any(zoom_mask):
            max_y_zoom = max(np.max(total_mc[zoom_mask]), np.max((this_data + data_eyhigh)[zoom_mask]))
            if is_logy:
                ax_inset.set_ylim(max(0.01, np.min(total_mc[zoom_mask])*0.1), max_y_zoom * 5)
            else:
                ax_inset.set_ylim(0.0, max_y_zoom * 1.5)

        # Formatting the inset
        ax_inset.tick_params(width=1, length=4, labelsize=8)
        for spine in ax_inset.spines.values():
            spine.set_linewidth(1.5)
        
        # Draw a box indicating the zoomed region on the main plot
        ax_main.indicate_inset_zoom(ax_inset, edgecolor="black", linewidth=1.5)

    # ==========================================
    # Ratio Plot
    # ==========================================
    fractional_mc_err = np.divide(mc_err, total_mc, out=np.zeros_like(mc_err), where=(total_mc != 0))
    mc_content_ratio = np.divide(total_mc, total_mc, out=np.zeros_like(total_mc), where=(total_mc != 0))
    
    ax_ratio.bar(bin_centers, 2 * fractional_mc_err, bottom=mc_content_ratio - fractional_mc_err,
                 width=np.diff(bin_edges), facecolor='none',
                 edgecolor='black', hatch='xxxx', linewidth=0.0,
                 label='Total Unc.', zorder=5)

    data_ratio = np.divide(this_data, total_mc, out=np.zeros_like(this_data, dtype=float), where=(total_mc != 0))
    data_ratio_eylow = np.divide(data_eylow, total_mc, out=np.zeros_like(data_eylow, dtype=float), where=(total_mc != 0))
    data_ratio_eyhigh = np.divide(data_eyhigh, total_mc, out=np.zeros_like(data_eyhigh, dtype=float), where=(total_mc != 0))

    ax_ratio.errorbar(bin_centers, data_ratio, 
                      yerr=np.vstack((data_ratio_eylow, data_ratio_eyhigh)), 
                      fmt='o', color='black', label='Data', 
                      markersize=5, capsize=3, linewidth=1.5, zorder=10)

    legend_labels_ratio = ["y=1", "Total Unc.", "Data/MC"]
    ax_ratio.legend(legend_labels_ratio, loc='upper left', fontsize=6, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

    #plt.tight_layout()
    # plt.savefig("./rate/" + obj['varcfg'].var_save_name + "_" + reg + "_data_mc_rate.pdf")
    plt.savefig("./rate/" + obj['varcfg'].var_save_name + "_" + reg + "_data_mc_rate.pdf", bbox_inches='tight')
    
    return fig, this_chi2, ndf

In [ ]:
def draw_uncertainty_contributions(obj, reg='sr', is_logx=False, sample_str="", region_str="Signal Region"):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

    if is_logx:
        ax.set_xscale('log')

    # Main plot styling
    ax.set_xlabel(obj['varcfg'].var_labels[0], fontsize=14)
    ax.set_ylabel("Fractional Uncertainty ($\delta N / N$)", fontsize=14)
    ax.tick_params(width=2, length=8, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # SBND text on top
    title_text = "SBND Internal, Total Unc." if not sample_str else f"SBND {sample_str}, Internal, Total Unc."
    ax.text(0.00, 1.02, title_text, transform=ax.transAxes, fontsize=14, fontweight='bold')
    ax.text(1.0, 1.02, region_str, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='right')

    # ==========================================
    # Data Preparation
    # ==========================================
    bin_edges = obj['varcfg'].bins
    ax.set_xlim(bin_edges[0], bin_edges[-1])

    # Fetch Data to calculate the Total MC denominator
    this_signal = obj[reg + '_signal_sel_reco_cv_rate']
    this_bkg = obj[reg + '_bkg_sel_reco_cv_rate']
    this_offbeam = obj[reg + '_offbeam_light_events']

    # Combine all MC and Offbeam for the denominator
    combined_mc = np.vstack((this_signal, this_bkg))
    full_mc_stack = np.vstack((combined_mc, this_offbeam))
    total_mc = np.sum(full_mc_stack, axis=0)

    # ==========================================
    # Uncertainty Extraction & Plotting
    # ==========================================
    # Dictionary mapping the legend label to your dictionary knob name
    cov_knobs = {
        "GENIE": "GENIE_NormCCCOH_0p5", 
        "Flux": "Flux", 
        "Geant4": "G4", 
        "MC Stat.": "MCstat",
        "POT": "POT",
        "Argon Target": "Ar",
        "Flat Detector": "Det"
    }

    total_variance = np.zeros_like(total_mc, dtype=float)
    
    # Define a set of safe colors (excluding black)
    safe_colors = plt.cm.tab10.colors
    color_idx = 0

    # 1. Loop through systematic and MC stat knobs
    for label, knob in cov_knobs.items():
        # Extract covariance matrix and get the diagonal (variance)
        cov_matrix = obj[f"{reg}_total_mc_evt_cov_{knob}"]['cov']
        variance = np.diag(cov_matrix)
        total_variance += variance
        
        # Calculate 1-sigma error and fraction
        err = np.sqrt(variance)
        frac_err = np.divide(err, total_mc, out=np.zeros_like(err), where=(total_mc != 0))
        
        # Plot as a step line using a safe color
        ax.step(bin_edges, list(frac_err) + [frac_err[-1]], where='post', 
                label=label, linewidth=2, color=safe_colors[color_idx % len(safe_colors)])
        color_idx += 1

    # 2. Add Offbeam (Data-driven background) uncertainty
    offbeam_cov = obj[f"{reg}_offbeam_light_events_cov"]
    offbeam_var = np.diag(offbeam_cov)
    total_variance += offbeam_var
    
    offbeam_err = np.sqrt(offbeam_var)
    offbeam_frac = np.divide(offbeam_err, total_mc, out=np.zeros_like(offbeam_err), where=(total_mc != 0))
    
    ax.step(bin_edges, list(offbeam_frac) + [offbeam_frac[-1]], where='post', 
            label="Offbeam Stat.", linewidth=2, color=safe_colors[color_idx % len(safe_colors)])

    # 3. Calculate and Plot the Total Uncertainty
    total_err = np.sqrt(total_variance)
    total_frac = np.divide(total_err, total_mc, out=np.zeros_like(total_err), where=(total_mc != 0))
    
    ax.step(bin_edges, list(total_frac) + [total_frac[-1]], where='post', 
            label="Total Uncertainty", color='black', linestyle='--', linewidth=2.5, zorder=10)

    # ==========================================
    # Final Formatting
    # ==========================================
    # Dynamically set y-limit to give 30% headroom above the highest total uncertainty bin
    max_y = np.max(total_frac)
    ax.set_ylim(0.0, max_y * 1.3)

    # Add legend
    ax.legend(loc='upper right', fontsize=10, frameon=False, ncol=3)

    fig.tight_layout()
    fig.savefig("./rate/" + obj['varcfg'].var_save_name + "_" + reg + "_uncertainty_contributions.pdf")

    return fig

In [ ]:
def draw_top_uncertainties_in_category(obj, total_knob_name, knob_list, reg='sr', region_str="Signal Region", is_logx=False, sample_str="", top_n=5):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

    if is_logx:
        ax.set_xscale('log')

    # Main plot styling
    ax.set_xlabel(obj['varcfg'].var_labels[0], fontsize=14)
    ax.set_ylabel("Fractional Uncertainty ($\delta N / N$)", fontsize=14)
    ax.tick_params(width=2, length=8, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # SBND text on top
    title_text = "SBND Internal, " + total_knob_name if not sample_str else f"SBND {sample_str}, Internal, " + total_knob_name
    ax.text(0.00, 1.02, title_text, transform=ax.transAxes, fontsize=14, fontweight='bold')
    ax.text(1.0, 1.02, region_str, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='right')

    # ==========================================
    # Data Preparation (Denominator)
    # ==========================================
    bin_edges = obj['varcfg'].bins
    ax.set_xlim(bin_edges[0], bin_edges[-1])

    this_signal = obj[reg + '_signal_sel_reco_cv_rate']
    this_bkg = obj[reg + '_bkg_sel_reco_cv_rate']
    this_offbeam = obj[reg + '_offbeam_light_events']

    combined_mc = np.vstack((this_signal, this_bkg))
    full_mc_stack = np.vstack((combined_mc, this_offbeam))
    total_mc = np.sum(full_mc_stack, axis=0)

    # ==========================================
    # Rank the Individual Knobs
    # ==========================================
    knob_frac_uncs = {}
    knob_impact = {}

    for knob in knob_list:
        cov_matrix = obj[f"{reg}_total_mc_evt_cov_{knob}"]['cov']
        variance = np.diag(cov_matrix)
        
        # Calculate fractional error for the actual plotting later
        err = np.sqrt(variance)
        frac_err = np.divide(err, total_mc, out=np.zeros_like(err), where=(total_mc != 0))
        knob_frac_uncs[knob] = frac_err
        
        # Rank by the sum of the absolute variance (diagonal components)
        knob_impact[knob] = np.sum(variance)

    # Sort the dictionary by the absolute variance in descending order
    sorted_knobs = sorted(knob_impact, key=knob_impact.get, reverse=True)
    top_knobs = sorted_knobs[:top_n]

    # ==========================================
    # Plotting Individual Knobs
    # ==========================================
    # Grab the tab10 color cycle (guaranteed not to include black)
    safe_colors = plt.cm.tab10.colors

    for i, knob in enumerate(top_knobs):
        frac_err = knob_frac_uncs[knob]
        
        # Clean up the label name
        clean_label = knob.replace("GENIEReWeight_SBN_v1_multisim_", "")
        clean_label = clean_label.replace("GENIEReWeight_SBN_v1_multisigma_", "")
        
        # Assign color from the safe_colors list
        ax.step(bin_edges, list(frac_err) + [frac_err[-1]], where='post', 
                label=clean_label, linewidth=2, color=safe_colors[i % len(safe_colors)])

    # ==========================================
    # Plotting Total Category Uncertainty
    # ==========================================
    total_cov = obj[f"{reg}_total_mc_evt_cov_{total_knob_name}"]['cov']
    total_var = np.diag(total_cov)
    total_err = np.sqrt(total_var)
    total_frac = np.divide(total_err, total_mc, out=np.zeros_like(total_err), where=(total_mc != 0))
    
    # Clean up total knob name for legend
    clean_total_label = "Total " + total_knob_name.replace("GENIE_NormCCCOH_0p5", "GENIE") 
    
    # Plot as a dashed black line over everything else (zorder=10)
    ax.step(bin_edges, list(total_frac) + [total_frac[-1]], where='post', 
            label=clean_total_label, color='black', linestyle='--', linewidth=2.5, zorder=10)

    # ==========================================
    # Final Formatting
    # ==========================================
    # Set y-limit based on the total envelope + 40% headroom for legend
    max_y = np.max(total_frac)
    ax.set_ylim(0.0, max_y * 1.4) 

    ax.legend(loc='upper right', fontsize=9, frameon=False, ncol=2)

    fig.tight_layout()

    fig.savefig("./rate/" + obj['varcfg'].var_save_name + "_" + reg + "_" + total_knob_name + "_frac_unc_tops.pdf")

    return fig

### workers

In [ ]:
def draw_plots_for_obj(obj, zoom_xrange=None, zoom_pos=[0.55, 0.35, 0.4, 0.4], data_vs_mc_xlim=None):
    for reg, reg_str in zip(['sr', 'sb'], ["Signal Region", "Sideband"]):
        draw_data_vs_mc_rate(obj, reg=reg, is_logx=False, is_logy=False, sample_str="", region_str=reg_str, xlim=data_vs_mc_xlim, ratio_ylim=(0.0, 2.0), zoom_xrange=zoom_xrange, zoom_pos=zoom_pos)
        draw_uncertainty_contributions(obj, reg=reg, sample_str="", region_str=reg_str)
        draw_top_uncertainties_in_category(obj, total_knob_name="GENIE_NormCCCOH_0p5", knob_list=genie_multisim_knobs + genie_multisigma_knobs, reg=reg, region_str=reg_str, is_logx=False, sample_str="", top_n=5)
        draw_top_uncertainties_in_category(obj, total_knob_name="Flux", knob_list=flux_knobs, reg=reg, region_str=reg_str, is_logx=False, sample_str="", top_n=5)
        draw_top_uncertainties_in_category(obj, total_knob_name="G4", knob_list=g4_knobs, reg=reg, region_str=reg_str, is_logx=False, sample_str="", top_n=5)

### run

In [ ]:
draw_plots_for_obj(p_mu_objs)
draw_plots_for_obj(cos_mu_objs, zoom_xrange=[0.9, 1.0], zoom_pos=[0.85, 0.15, 0.25, 0.5], data_vs_mc_xlim=[-1, 1.7])
draw_plots_for_obj(cos_pi_objs)